# ЛР3. Основной XML-RPC сервер

Сервер выполняет операции ЛР1 и ЛР2 на порту 8008. Клиент обращается к нему через
прокси. Отправка статистики перенесена в прокси, поэтому этот сервер работает
независимо от журнала. Запустите ячейку и оставьте её работающей.
Для остановки используйте прерывание ядра. Перед повторным запуском перезапустите ядро.


In [1]:
from pathlib import Path
from xmlrpc.server import SimpleXMLRPCRequestHandler, SimpleXMLRPCServer
import datetime
import pickle
import xmlrpc.client

import pandas as pd


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


blacklist_file = Path.cwd() / 'bad_boys2.csv'


# ЛР2, чёрный список: сравнивать без учёта регистра; Семён и Семен равнозначны.
def normalize_text(name_part):
    # Регистр и различие между буквами ё и е при проверке не учитываются.
    return str(name_part).strip().lower().replace('ё', 'е')


# ЛР2, чёрный список: проверка отличия на одну букву.
# Реализация учитывает замену, пропуск или добавление буквы.
def differs_by_one_letter(first_value, second_value):
    first_value = normalize_text(first_value)
    second_value = normalize_text(second_value)
    if abs(len(first_value) - len(second_value)) > 1:
        return False
    # При одинаковой длине допускается только одна замененная буква.
    if len(first_value) == len(second_value):
        return sum(first_letter != second_letter for first_letter, second_letter in zip(first_value, second_value)) == 1
    # При разной длине проверяется одна добавленная или пропущенная буква.
    if len(first_value) > len(second_value):
        first_value, second_value = second_value, first_value
    for letter_index in range(len(first_value)):
        if first_value[letter_index] != second_value[letter_index]:
            return first_value[letter_index:] == second_value[letter_index + 1:]
    return True


# ЛР2, чёрный список: вернуть yes, similar со списком похожих записей либо no.
# Принята трактовка: на одну букву отличается одно поле ФИО, остальные совпадают.
def black_list_check(surname, name, patronym, birth):
    blacklist_table = pd.read_csv(blacklist_file, encoding='utf-8-sig')
    entered_person = {
        'surname': str(surname).strip(),
        'name': str(name).strip(),
        'patronym': str(patronym).strip(),
        'birth': str(birth).strip(),
    }
    similar_people = []
    exact_match = False

    for _, blacklist_person in blacklist_table.iterrows():
        # Дата рождения остается точным условием из первой лабораторной работы.
        if str(blacklist_person['Birth']).strip() != entered_person['birth']:
            continue
        entered_names = [entered_person['surname'], entered_person['name'], entered_person['patronym']]
        blacklist_names = [blacklist_person['Surname'], blacklist_person['Name'], blacklist_person['Patronym']]
        different_fields = []
        for entered_value, blacklist_value in zip(entered_names, blacklist_names):
            if normalize_text(entered_value) != normalize_text(blacklist_value):
                different_fields.append((entered_value, blacklist_value))

        if not different_fields:
            exact_match = True
            break
        # Отличаться на одну букву может только одно поле ФИО.
        if len(different_fields) == 1 and differs_by_one_letter(*different_fields[0]):
            similar_people.append({
                'surname': str(blacklist_person['Surname']).strip(),
                'name': str(blacklist_person['Name']).strip(),
                'patronym': str(blacklist_person['Patronym']).strip(),
                'birth': str(blacklist_person['Birth']).strip(),
            })

    blacklist_check_result = entered_person.copy()
    if exact_match:
        blacklist_check_result['status'] = 'yes'
    elif similar_people:
        blacklist_check_result['status'] = 'similar'
        blacklist_check_result['similar'] = similar_people
    else:
        blacklist_check_result['status'] = 'no'

    return blacklist_check_result


def deserialize_image(binary_image):
    return pickle.loads(binary_image.data)


def serialize_image(image_array):
    return xmlrpc.client.Binary(pickle.dumps(image_array, protocol=pickle.HIGHEST_PROTOCOL))


def get_channel_count(image_array):
    if image_array.ndim == 2:
        return 1
    if image_array.ndim == 3 and image_array.shape[2] in (1, 3):
        return image_array.shape[2]
    raise ValueError('Ожидается изображение (M,N), (M,N,1) или (M,N,3)')


def get_pixel_brightness(image_array, row_index, column_index, channel_count):
    if image_array.ndim == 2:
        return int(image_array[row_index, column_index])
    if channel_count == 1:
        return int(image_array[row_index, column_index, 0])
    red = int(image_array[row_index, column_index, 0])
    green = int(image_array[row_index, column_index, 1])
    blue = int(image_array[row_index, column_index, 2])
    # Яркость цветного пикселя определяется средним значением трех каналов.
    return (red + green + blue) // 3


# ЛР2, бинаризация: вместе с изображением вычислить долю пикселей строго выше порога.
def binarize_array(image_array, threshold):
    channel_count = get_channel_count(image_array)
    height, width = image_array.shape[:2]
    pixels_above_threshold = 0
    # Пиксели обходятся вручную, так как библиотечная бинаризация запрещена.
    for row_index in range(height):
        for column_index in range(width):
            brightness = get_pixel_brightness(image_array, row_index, column_index, channel_count)
            binary_value = 255 if brightness > threshold else 0
            if binary_value == 255:
                pixels_above_threshold += 1
            if image_array.ndim == 2:
                image_array[row_index, column_index] = binary_value
            else:
                for channel_index in range(channel_count):
                    image_array[row_index, column_index, channel_index] = binary_value
    # Процент считается по количеству пикселей, а не по числу цветовых каналов.
    above_threshold_percent = pixels_above_threshold * 100 / (height * width)
    return image_array, above_threshold_percent


# Базовые методы ЛР1. В ЛР3 все события регистрируются в прокси.
def ping():
    return True


def now():
    return datetime.datetime.now()


def show_type(argument):
    return str(argument), str(type(argument)), argument


def test_sum(first_number, second_number):
    return first_number + second_number


def test_pow(base_number, exponent):
    return base_number ** exponent


def color_inversion(binary_image):
    image_array = deserialize_image(binary_image)
    channel_count = get_channel_count(image_array)
    height, width = image_array.shape[:2]
    # Инверсия выполняется вручную отдельно для каждого пикселя и канала.
    for row_index in range(height):
        for column_index in range(width):
            if image_array.ndim == 2:
                image_array[row_index, column_index] = 255 - int(image_array[row_index, column_index])
            else:
                for channel_index in range(channel_count):
                    image_array[row_index, column_index, channel_index] = 255 - int(image_array[row_index, column_index, channel_index])
    return serialize_image(image_array)


def binarize_image(binary_image, threshold):
    threshold = int(threshold)
    if threshold < 1 or threshold > 255:
        raise ValueError('Порог должен находиться в диапазоне 1..255')
    binarized_image, above_threshold_percent = binarize_array(deserialize_image(binary_image), threshold)
    return serialize_image(binarized_image)


# ЛР2: новая XML-RPC функция возвращает бинарное изображение и процент пикселей выше порога.
def binarize_with_coverage(binary_image, threshold):
    threshold = int(threshold)
    if threshold < 1 or threshold > 255:
        raise ValueError('Порог должен находиться в диапазоне 1..255')
    binarized_image, above_threshold_percent = binarize_array(deserialize_image(binary_image), threshold)
    return {
        'image': serialize_image(binarized_image),
        'above_threshold_percent': above_threshold_percent,
        'threshold': threshold,
    }


def vertical_flip(binary_image):
    image_array = deserialize_image(binary_image)
    channel_count = get_channel_count(image_array)
    height, width = image_array.shape[:2]
    # Меняются местами симметричные столбцы левой и правой половины.
    for row_index in range(height):
        for column_index in range(width // 2):
            opposite_column_index = width - column_index - 1
            if image_array.ndim == 2:
                temporary_value = image_array[row_index, column_index]
                image_array[row_index, column_index] = image_array[row_index, opposite_column_index]
                image_array[row_index, opposite_column_index] = temporary_value
            else:
                for channel_index in range(channel_count):
                    temporary_value = image_array[row_index, column_index, channel_index]
                    image_array[row_index, column_index, channel_index] = image_array[row_index, opposite_column_index, channel_index]
                    image_array[row_index, opposite_column_index, channel_index] = temporary_value
    return serialize_image(image_array)


# Публикация методов основного сервиса, включая новую функцию из ЛР2.
server = SimpleXMLRPCServer(('127.0.0.1', 8008), requestHandler=RequestHandler, allow_none=True, logRequests=True)
server.register_introspection_functions()
server.register_function(ping, 'ping')
server.register_function(now, 'now')
server.register_function(show_type, 'type')
server.register_function(test_sum, 'sum')
server.register_function(test_pow, 'pow')
server.register_function(black_list_check, 'black_list_check')
server.register_function(color_inversion, 'color_inversion')
server.register_function(binarize_image, 'binarize_image')
server.register_function(binarize_with_coverage, 'binarize_with_coverage')
server.register_function(vertical_flip, 'vertical_flip')

print('Основной XML-RPC сервер слушает порт 8008')
try:
    server.serve_forever()
except KeyboardInterrupt:
    print("Сервер остановлен")
finally:
    server.server_close()


Основной XML-RPC сервер слушает порт 8008
Основной XML-RPC сервер слушает порт 8008
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:57] "POST /RPC2 HTTP/1.1" 200 -
Основной XML-RPC сервер слушает порт 8008
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:54] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17